# 🗺️ 서울시 통합 재난 위험 지수 (IDRI) — 데이터 분석 리포트

> **분석 목표**: 서울 25개 자치구의 홍수 취약성 · 이동 장애 인구 · 복지 시설 접근성을 결합하여  
> 자치구별 재난 위험 순위를 산출하고, 고위험 핫스팟을 식별합니다.

| 항목 | 내용 |
|------|------|
| 분석 지역 | 서울특별시 25개 자치구 |
| 기준 연도 | 2024 |
| 주요 도구 | GeoPandas · Matplotlib · Seaborn · sklearn |
| 데이터 출처 | 국토지리정보원, 서울 열린데이터광장 |


---
## ⚙️ 0. 환경 설정

한글 폰트 설치 및 필수 라이브러리 임포트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 한글 폰트 설치 (Colab 런타임당 1회)
import subprocess
subprocess.run(['apt-get', 'install', '-y', '-q', 'fonts-nanum'], capture_output=True)

import os, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.font_manager as fm
import seaborn as sns
from shapely.geometry import Point
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings('ignore')

# 폰트 등록
font_path = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc('font', family='NanumBarunGothic')
plt.rcParams['axes.unicode_minus'] = False
print("✅ 환경 설정 완료")


---
## 📂 1. 사용 데이터 소개

| 파일명 | 유형 | 주요 컬럼 | 용도 |
|--------|------|----------|------|
| `N3L_F001.shp` | 벡터 · 라인 | geometry | 등고선 배경 지도 |
| `N3P_F002.shp` | 벡터 · 포인트 | HEIGHT, geometry | 표고점 — 고도 분포 분석 핵심 |
| `침수흔적도 최종2.shp` | 벡터 · 폴리곤 | geometry | 침수 이력 영역 — 홍수 취약성 산출 |
| `BND_ADM_DONG_PG.shp` | 벡터 · 폴리곤 | ADM_CD, ADM_NM | 행정동 경계 → 자치구 단위 집계 |
| `장애인현황.csv` | 테이블 | 자치구, 지체, 뇌병변 | 이동 취약성 비율 산출 |
| `재가노인복지시설.csv` | 테이블 | 시설주소, 경도, 위도 | 시설 접근성 (취약성 역수) 산출 |

> ⚠️ **좌표계 주의**: 각 SHP가 서로 다른 CRS로 배포되므로, 공간 조인 전 전원 **EPSG:5174** 로 리프로젝션이 필수입니다.


---
## 📥 2. 데이터 로드 & CRS 통일

In [ ]:
BASE = '/content/drive/MyDrive/Colab Notebooks/'

# ── 공간 데이터
gdf   = gpd.read_file(BASE + 'N3L_F001.shp')
elev  = gpd.read_file(BASE + 'N3P_F002.shp')
flood = gpd.read_file(BASE + '침수흔적도 최종2.shp')
dong  = gpd.read_file(BASE + 'BND_ADM_DONG_PG.shp')

# ── 통계 데이터
df_disabled_age_raw    = pd.read_csv(BASE + '장애인+현황(장애유형별_연령별)_20260509122444.csv',
                                     encoding='utf-8', header=[0, 1, 2])
df_disabled_status_raw = pd.read_csv(BASE + '서울시 장애유형별. 장애정도별 장애인 등록현황.csv',
                                     encoding='utf-8', header=[0, 1])
df_welfare             = pd.read_csv(BASE + '서울시 사회복지시설(재가노인복지시설) 목록.csv',
                                     encoding='cp949')

# ── CRS 통일 (EPSG:5174)
for name, gdf_ in [('elev', elev), ('flood', flood), ('dong', dong)]:
    if gdf_.crs is None or gdf_.crs.to_epsg() != 5174:
        gdf_.to_crs('EPSG:5174', inplace=True)

# ── 서울 행정동 추출 (ADM_CD 11xxx)
seoul_dong = dong[dong['ADM_CD'].astype(str).str.startswith('11')].copy()
if seoul_dong.crs.to_epsg() != 5174:
    seoul_dong = seoul_dong.to_crs('EPSG:5174')

print(f"표고점:    {len(elev):>6,}개")
print(f"침수 폴리곤: {len(flood):>4,}개")
print(f"서울 행정동: {len(seoul_dong):>4,}개")
print("✅ 데이터 로드 완료")


---
## 🗺️ 3. 분석 플로우

```
[STEP 1] 지형 탐색        → 등고선 · 표고 지도 시각화
[STEP 2] 침수 오버레이    → 표고 + 침수 흔적 중첩 지도
[STEP 3] 공간 조인        → 침수 영역 내/외 표고점 분리
[STEP 4] 고도 분포 분석   → 히스토그램 · 박스플롯 비교
[STEP 5] 3대 지표 산출    → 이동 · 홍수 · 시설 취약성
[STEP 6] IDRI 산출        → 가중 합산 → 히트맵 · 분해 차트
[STEP 7] 핫스팟 식별      → 60 퍼센타일 기준 복합 고위험 구 도출
```

---
### 📊 Chart 1 — 표고 + 침수 흔적 오버레이 지도

**인사이트**: 침수 흔적이 저지대(표고 낮은 영역)에 집중 분포하는지 육안으로 확인합니다.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))

elev.plot(
    column='HEIGHT', cmap='terrain', alpha=0.7, ax=ax, legend=True,
    legend_kwds={'label': '표고 (m)', 'orientation': 'vertical', 'shrink': 0.6}
)
flood.plot(color='red', edgecolor='red', alpha=0.4, ax=ax)

ax.legend(
    handles=[mpatches.Patch(color='red', alpha=0.4, label='침수 흔적')],
    loc='upper left', fontsize=12
)
ax.set_title('표고 + 침수 흔적 오버레이 지도', fontsize=16, fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.show()


---
### 🔗 공간 조인 — 침수 영역 내/외 표고점 분리

`gpd.sjoin(predicate='within')` 으로 침수 폴리곤 내부에 포함된 표고점과 그렇지 않은 표고점을 분리합니다.


In [ ]:
# 침수 영역 내 표고점
elev_in_flood     = gpd.sjoin(elev, flood, how='inner', predicate='within')
flooded_idx       = elev_in_flood.index.unique()
elev_not_in_flood = elev[~elev.index.isin(flooded_idx)].copy()

print(f"침수 영역 내  표고점: {len(elev_in_flood):,}개")
print(f"침수 영역 외  표고점: {len(elev_not_in_flood):,}개")
print(f"전체 표고점:         {len(elev):,}개")


---
### 📊 Chart 2 — 침수 영역 내 표고 분포 히스토그램

**인사이트**: 침수가 발생한 지점의 표고 분포 — 어느 고도 범위에서 침수가 빈발하는지 파악합니다.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

sns.histplot(elev_in_flood['HEIGHT'], kde=True, ax=ax, bins=25,
             color='steelblue', label='침수 영역 내 표고')

ax.set_title('침수 흔적 영역 내 표고 분포', fontsize=15, fontweight='bold')
ax.set_xlabel('표고 (m)', fontsize=12)
ax.set_ylabel('포인트 수', fontsize=12)
ax.grid(axis='y', linestyle='--', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.show()

print("\n📋 침수 영역 내 표고 기술통계:")
print(elev_in_flood['HEIGHT'].describe().round(2).to_string())


---
### 📊 Chart 3 — 침수 여부별 표고 분포 비교 (박스플롯)

**인사이트**: 침수 지역 vs 비침수 지역의 표고 중앙값 차이를 통계적으로 비교합니다.  
중앙값 기준으로 **침수 지역이 비침수 지역보다 약 25m 낮음**을 확인할 수 있습니다.


In [ ]:
df_cmp = pd.concat([
    elev_in_flood[['HEIGHT']].assign(구분='침수 지역'),
    elev_not_in_flood[['HEIGHT']].assign(구분='비침수 지역')
])

fig, ax = plt.subplots(figsize=(9, 6))
sns.boxplot(
    x='구분', y='HEIGHT', data=df_cmp, ax=ax,
    palette={'침수 지역': 'steelblue', '비침수 지역': 'salmon'},
    order=['침수 지역', '비침수 지역']
)
ax.set_title('침수 여부별 표고 분포 비교', fontsize=15, fontweight='bold')
ax.set_xlabel('지역 구분', fontsize=12)
ax.set_ylabel('표고 (m)', fontsize=12)
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

print("\n📋 침수 지역 표고 통계:")
print(elev_in_flood['HEIGHT'].describe().round(2).to_string())
print("\n📋 비침수 지역 표고 통계:")
print(elev_not_in_flood['HEIGHT'].describe().round(2).to_string())


---
## 🧮 4. IDRI 산출 — 3대 취약성 지표 계산

### 산출 공식

$$\text{IDRI} = 0.4 \times \text{이동취약성} + 0.4 \times \text{홍수취약성} + 0.2 \times \text{시설취약성}$$

| 지표 | 가중치 | 산출 방법 |
|------|--------|----------|
| 이동 취약성 | 0.4 | 지체·뇌병변 장애 인구 비율 (%) ÷ 100 |
| 홍수 취약성 | 0.4 | 자치구별 침수 흔적 면적 log₁(1+x) 후 MinMax 정규화 |
| 시설 취약성 | 0.2 | 복지 시설 밀도 역수 (MinMax 정규화) |

> **가중치 근거**: 이동 취약성과 홍수 취약성은 각각 재난 시 자력 대피 불가 인구와 직접 침수 위험을 나타내므로 동등 최대 가중. 시설 취약성은 위경도 데이터 누락으로 0.2로 하향 조정.


In [ ]:
# ── 자치구 코드 매핑
gu_code_to_name = {
    '11010':'종로구','11020':'중구','11030':'용산구','11040':'성동구','11050':'광진구',
    '11060':'동대문구','11070':'중랑구','11080':'성북구','11090':'강북구','11100':'도봉구',
    '11110':'노원구','11120':'은평구','11130':'서대문구','11140':'마포구','11150':'양천구',
    '11160':'강서구','11170':'구로구','11180':'금천구','11190':'영등포구','11200':'동작구',
    '11210':'관악구','11220':'서초구','11230':'강남구','11240':'송파구','11250':'강동구'
}

seoul_dong_proc = seoul_dong.copy()
seoul_dong_proc['GU_CODE'] = seoul_dong_proc['ADM_CD'].astype(str).str[:5]
seoul_dong_proc['자치구']   = seoul_dong_proc['GU_CODE'].map(gu_code_to_name)
seoul_dong_proc.dropna(subset=['자치구'], inplace=True)

seoul_gu_gdf = seoul_dong_proc.dissolve(by='자치구', aggfunc='first').reset_index()
if seoul_gu_gdf.crs.to_epsg() != 5174:
    seoul_gu_gdf = seoul_gu_gdf.to_crs('EPSG:5174')

# ── 이동 취약성 (지체·뇌병변 비율)
new_col = []
for l0, l1 in df_disabled_status_raw.columns:
    if '자치구' in l0: new_col.append('자치구')
    elif l0.strip()=='합계' and l1.strip()=='합계': new_col.append('전체_합계')
    else: new_col.append(f"{l0.strip()}_{l1.strip()}")
df_disabled_status_raw.columns = new_col
ds = df_disabled_status_raw.iloc[1:].copy()
for c in ds.columns:
    if c == '자치구': continue
    ds[c] = pd.to_numeric(ds[c].astype(str).str.replace(',','',regex=False).str.strip(), errors='coerce')
ds.dropna(subset=['자치구'], inplace=True)
num_cols = ds.select_dtypes(include='number').columns.tolist()
ds['Total'] = ds[num_cols].sum(axis=1)
mob_cols = [c for c in num_cols if '지체' in c or '뇌병변' in c]
ds['Mobility_Impaired'] = ds[mob_cols].sum(axis=1)
ds['Mobility_Impaired_Proportion'] = (ds['Mobility_Impaired'] / ds['Total'].replace(0, np.nan)) * 100
ds['Mobility_Impaired_Proportion'].fillna(0, inplace=True)

# ── 홍수 취약성 (침수 면적 log 변환)
inter = gpd.overlay(seoul_dong_proc, flood, how='intersection')
inter['area_sqm'] = inter.geometry.area
flooded_by_gu = inter.groupby('자치구')['area_sqm'].sum().reset_index().rename(columns={'area_sqm':'Flood_Area'})
flooded_by_gu['Flood_Vulnerability_Score'] = np.log1p(flooded_by_gu['Flood_Area'])

# ── 기본 프레임 합산
seoul_gu_analysis = seoul_gu_gdf.merge(ds[['자치구','Mobility_Impaired_Proportion']], on='자치구', how='left')
seoul_gu_analysis = seoul_gu_analysis.merge(flooded_by_gu[['자치구','Flood_Vulnerability_Score']], on='자치구', how='left')
seoul_gu_analysis.fillna({'Mobility_Impaired_Proportion':0,'Flood_Vulnerability_Score':0}, inplace=True)
seoul_gu_analysis['Area_sqkm'] = seoul_gu_analysis.geometry.area / 1_000_000

# ── 시설 취약성 (위경도 있을 경우만)
df_welfare.dropna(subset=['시설주소'], inplace=True)
if '경도' in df_welfare.columns and '위도' in df_welfare.columns:
    gdf_w = gpd.GeoDataFrame(df_welfare,
        geometry=[Point(xy) for xy in zip(df_welfare['경도'], df_welfare['위도'])],
        crs='EPSG:4326').to_crs('EPSG:5174')
    fc = gdf_w.sjoin(seoul_gu_gdf, how='inner', predicate='within').groupby('자치구').size().reset_index(name='시설개수')
else:
    print("⚠️  경도/위도 컬럼 없음 → 시설 취약성 0으로 처리")
    fc = pd.DataFrame({'자치구': seoul_gu_analysis['자치구'], '시설개수': 0})

seoul_gu_analysis = seoul_gu_analysis.merge(fc, on='자치구', how='left')
seoul_gu_analysis['시설개수'].fillna(0, inplace=True)
seoul_gu_analysis['Facility_Density'] = seoul_gu_analysis['시설개수'] / seoul_gu_analysis['Area_sqkm'].replace(0, np.nan)
seoul_gu_analysis['Facility_Density'].fillna(0, inplace=True)

scaler = MinMaxScaler()
seoul_gu_analysis['Facility_Vulnerability_Score'] = 1 - scaler.fit_transform(seoul_gu_analysis[['Facility_Density']])

# ── 정규화 & IDRI 계산
seoul_gu_analysis['Mobility_Vulnerability_Score']         = seoul_gu_analysis['Mobility_Impaired_Proportion'] / 100
seoul_gu_analysis['Normalized_Flood_Vulnerability_Score'] = scaler.fit_transform(seoul_gu_analysis[['Flood_Vulnerability_Score']])

W_M, W_F, W_FA = 0.4, 0.4, 0.2
seoul_gu_analysis['IDRI_Score'] = (
    W_M  * seoul_gu_analysis['Mobility_Vulnerability_Score'] +
    W_F  * seoul_gu_analysis['Normalized_Flood_Vulnerability_Score'] +
    W_FA * seoul_gu_analysis['Facility_Vulnerability_Score']
)
seoul_gu_analysis['Mobility_Contribution'] = W_M  * seoul_gu_analysis['Mobility_Vulnerability_Score']
seoul_gu_analysis['Flood_Contribution']    = W_F  * seoul_gu_analysis['Normalized_Flood_Vulnerability_Score']
seoul_gu_analysis['Facility_Contribution'] = W_FA * seoul_gu_analysis['Facility_Vulnerability_Score']

print("✅ IDRI 산출 완료")
print(f"\n📋 IDRI 상위 5개 자치구:")
print(seoul_gu_analysis.nlargest(5,'IDRI_Score')[['자치구','IDRI_Score','Mobility_Contribution','Flood_Contribution','Facility_Contribution']].round(3).to_string(index=False))


---
### 📊 Chart 4 — 서울시 행정구역별 IDRI 히트맵

**인사이트**: 북부(강북·은평·도봉·노원)와 서부(강서)가 상대적으로 고위험 — 한강 지류 저지대 및 구릉 지형이 침수 취약성을 높입니다.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 14))

seoul_gu_analysis.plot(
    column='IDRI_Score', cmap='YlGnBu', linewidth=0.8, ax=ax, edgecolor='0.6',
    legend=True,
    legend_kwds={'label': 'IDRI 점수', 'orientation': 'horizontal', 'shrink': 0.6, 'pad': 0.01}
)

for _, row in seoul_gu_analysis.iterrows():
    c = row.geometry.centroid
    ax.annotate(
        f"{row['자치구']}\n{row['IDRI_Score']:.3f}",
        xy=(c.x, c.y), ha='center', va='center',
        fontsize=7, color='black',
        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.5, ec='none')
    )

ax.set_title('서울시 행정구역별 통합 재난 위험 지수 (IDRI) 히트맵', fontsize=17, fontweight='bold', pad=16)
ax.set_axis_off()
plt.tight_layout()
plt.show()


---
### 📊 Chart 5 — IDRI 점수 분해 차트 (기여도)

**인사이트**: 상위 5개 구 전부에서 **홍수 기여도가 1위** — 이동 취약성만 높아도 홍수 취약성이 낮으면 IDRI는 낮게 산출됩니다.


In [ ]:
df_contrib = (
    seoul_gu_analysis
    .sort_values('IDRI_Score', ascending=False)
    .set_index('자치구')[['Mobility_Contribution','Flood_Contribution','Facility_Contribution']]
)

fig, ax = plt.subplots(figsize=(12, 10))
df_contrib.plot(
    kind='barh', stacked=True,
    color=['#E07B7B','#4A90D9','#5DBE8A'],
    ax=ax
)
ax.set_title('행정구역별 IDRI 점수 분해 (가중치 기여도)', fontsize=15, fontweight='bold')
ax.set_xlabel('IDRI 기여도', fontsize=12)
ax.set_ylabel('자치구', fontsize=12)
ax.legend(
    title='위험 요소',
    labels=['이동 취약성 (×0.4)', '홍수 취약성 (×0.4)', '시설 취약성 (×0.2)'],
    bbox_to_anchor=(1.02, 1), loc='upper left'
)
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


---
## 📋 5. 분석 결과 테이블

### IDRI 전체 순위 (25개 자치구)


In [ ]:
result_table = (
    seoul_gu_analysis[['자치구','IDRI_Score','Mobility_Contribution','Flood_Contribution','Facility_Contribution']]
    .sort_values('IDRI_Score', ascending=False)
    .reset_index(drop=True)
)
result_table.index = result_table.index + 1
result_table.index.name = '순위'
result_table.columns = ['자치구','IDRI 점수','이동 기여','홍수 기여','시설 기여']

# 주요 기여 요인
def main_cause(row):
    d = {'이동':row['이동 기여'],'홍수':row['홍수 기여'],'시설':row['시설 기여']}
    return max(d, key=d.get)
result_table['주요 요인'] = result_table.apply(main_cause, axis=1)
result_table = result_table.round(3)
display(result_table)


---
## 🔴 6. 핫스팟 식별 — 복합 고위험 자치구

### 핫스팟 정의 기준

> **홍수 취약성** AND **이동 취약성** 모두 **60 퍼센타일 초과**  
> → 침수 발생 시 자력 대피가 어려운 인구가 집중된 '복합 위험' 구

침수가 발생했을 때 가장 큰 피해를 입을 수 있는 구를 우선 개입 대상으로 식별합니다.


In [ ]:
f_thr = seoul_gu_analysis['Normalized_Flood_Vulnerability_Score'].quantile(0.60)
m_thr = seoul_gu_analysis['Mobility_Vulnerability_Score'].quantile(0.60)

print(f"홍수 취약성 60 퍼센타일 임계치: {f_thr:.4f}")
print(f"이동 취약성 60 퍼센타일 임계치: {m_thr:.4f}")

seoul_gu_analysis['Hotspot'] = (
    (seoul_gu_analysis['Normalized_Flood_Vulnerability_Score'] > f_thr) &
    (seoul_gu_analysis['Mobility_Vulnerability_Score'] > m_thr)
)

hotspot = seoul_gu_analysis[seoul_gu_analysis['Hotspot']]
print(f"\n🔴 핫스팟 자치구 ({len(hotspot)}개): {', '.join(hotspot['자치구'].tolist())}")
print("\n📋 핫스팟 상세:")
display(hotspot[['자치구','IDRI_Score','Mobility_Vulnerability_Score',
                 'Normalized_Flood_Vulnerability_Score','Flood_Contribution','Mobility_Contribution']]
        .sort_values('IDRI_Score', ascending=False).round(3))


---
### 📊 Chart 6 — 홍수 × 이동 취약성 핫스팟 지도

**인사이트**: 노원·도봉·성북·서대문구 4개 구가 두 위험이 동시에 높은 복합 취약 지역으로 확인됩니다.  
서대문구는 IDRI 상위 5위에도 포함되어 특히 시급한 대응이 요구됩니다.


In [ ]:
fig, ax = plt.subplots(figsize=(14, 14))

# 기본 배경
seoul_gu_analysis.plot(ax=ax, color='#e8e8e8', edgecolor='0.6', linewidth=0.8, alpha=0.8)

# 핫스팟 강조
if not hotspot.empty:
    hotspot.plot(ax=ax, color='#D94444', edgecolor='#8B1A1A', linewidth=1.2, alpha=0.9)

# 구 이름 레이블
for _, row in seoul_gu_analysis.iterrows():
    c = row.geometry.centroid
    is_hot = row['Hotspot']
    ax.annotate(
        row['자치구'],
        xy=(c.x, c.y), ha='center', va='center',
        fontsize=9 if is_hot else 7.5,
        fontweight='bold' if is_hot else 'normal',
        color='white' if is_hot else '#333333'
    )

ax.legend(handles=[
    mpatches.Patch(color='#D94444', label='🔴 고위험 핫스팟 (홍수 + 이동 취약 동시 高)'),
    mpatches.Patch(color='#e8e8e8', ec='gray', label='일반 지역')
], loc='upper left', fontsize=11)

ax.set_title('서울시 홍수 × 이동 취약성 핫스팟 지도 (60 퍼센타일 기준)',
             fontsize=17, fontweight='bold', pad=16)
ax.set_axis_off()
plt.tight_layout()
plt.show()


---
## 📌 7. 결론 & 다음 단계

### 핵심 발견

| 항목 | 내용 |
|------|------|
| **IDRI 최고위험 구** | 강북구 (0.589) |
| **상위 5개 구** | 강북구, 은평구, 강서구, 종로구, 서대문구 |
| **모든 상위구 공통 요인** | 홍수 취약성이 1위 기여 요인 |
| **복합 고위험 핫스팟** | 노원구, 도봉구, 서대문구, 성북구 (4개 구) |
| **침수 고도 패턴** | 침수 흔적 표고점의 75% 이상이 해발 25m 이하 |

### 다음 단계 제안

| 우선순위 | 과제 | 기대 효과 |
|----------|------|----------|
| 🔴 P1 | 복지 시설 주소 지오코딩 (Kakao/Naver API) | 시설 취약성 지표 완결 → IDRI 정밀도 향상 |
| 🔴 P1 | 핫스팟 4구 행정동 단위 세분화 분석 | 구 내 고위험 동 식별 → 국지적 개입 가능 |
| 🟡 P2 | 기상청 최근 10년 강수 데이터 통합 | 홍수 취약성을 이력 기반 → 예측 기반으로 전환 |
| 🟡 P2 | 고령 인구 비율 추가 변수 검토 | 이동 취약성 지표 다양화 |
| 🟢 P3 | Folium / Kepler.gl 인터랙티브 지도 제작 | 결과 공유 및 정책 커뮤니케이션 강화 |
| 🟢 P3 | 연도별 IDRI 시계열 비교 (2020–2024) | 취약성 변화 추이 모니터링 |
